In [1]:
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
!sudo apt-get install nodejs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  javascript-common libc-ares2 libjs-highlight.js libnode72 nodejs-doc
Suggested packages:
  apache2 | lighttpd | httpd npm
The following NEW packages will be installed:
  javascript-common libc-ares2 libjs-highlight.js libnode72 nodejs nodejs-doc
0 upgraded, 6 newly installed, 0 to remove and 2 not upgraded.
Need to get 13.7 MB of archives.
After this operation, 54.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 javascript-common all 11+nmu1 [5,936 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjs-highlight.js all 9.18.5+dfsg1-1 [367 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libc-ares2 amd64 1.18.1-1ubuntu0.22.04.3 [45.1 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libnode72 amd64 12.22.9~dfsg-1ubuntu3.6 [10.8 MB]
G

In [3]:
!pip install librosa soundfile language-tool-python yt-dlp textstat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.2 MB/s eta 0:00:00


In [ ]:
import torch
import textstat
import yt_dlp
import pandas as pd
import language_tool_python
from transformers import pipeline

# ==============================
# 1. إعداد الجهاز (GPU if available)
# ==============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"System Running on: {device.upper()}")

# ==============================
# 2. تحميل الموديلات
# ==============================
# Whisper Tiny: سريع جداً ومناسب للـ CPU في Colab
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=device,
    generate_kwargs={"task": "transcribe"}
)

# LanguageTool: لتقييم القواعد اللغوية (Grammar & Style)
tool = language_tool_python.LanguageTool('en-US')

# ==============================
# 3. دالة تحميل الصوت من اليوتيوب
# ==============================
def download_youtube_audio(url):
    print(f"Downloading from: {url}")
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'interview_audio.%(ext)s',
        'quiet': True,
        'no_warnings': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.prepare_filename(info)

# ==============================
# 4. التقييم الشامل (The Master Function)
# ==============================
def assess_clarity_final(audio_path):
    print("Step 1: Transcribing audio to text...")
    result = asr_pipeline(audio_path, return_timestamps=True)
    text = result["text"].strip()

    if not text:
        return {"Error": "No clear speech detected in the audio."}

    print("Step 2: Analyzing Grammar & Structure...")
    matches = tool.check(text)
    num_errors = len(matches)
    word_count = max(len(text.split()), 1)

    grammar_score = max(0, 100 - (num_errors / word_count * 100))
    error_categories = list(set([m.category for m in matches]))[:3] # أهم 3 أنواع أخطاء

    print("Step 3: Evaluating Vocabulary Quality...")
    # Dale-Chall: 0-10 (بسيط لمتطور). نضرب في 10 لنجعله من 100
    vocab_raw = textstat.dale_chall_readability_score(text)
    vocab_score = min(vocab_raw * 10, 100)

    final_score = (grammar_score * 0.7) + (vocab_score * 0.3)

    if final_score >= 85:
        level = "High Clarity (Professional & Structured)"
    elif final_score >= 70:
        level = "Moderate Clarity (Understandable but has minor issues)"
    else:
        level = "Low Clarity (Unstructured or many language errors)"

    return {
        "Candidate Speech (Sample)": text[:-1] + "...",
        "Grammar Accuracy": f"{grammar_score:.1f}%",
        "Grammar Issues Found": num_errors,
        "Common Errors": ", ".join(error_categories) if error_categories else "None",
        "Vocabulary Score": f"{vocab_score:.1f}/100",
        "FINAL CLARITY INDEX": f"{final_score:.1f}/100",
        "OVERALL ASSESSMENT": level
    }

# ==============================
# 5. تشغيل التجربة
# ==============================
try:
    url = "https://www.youtube.com/watch?v=gwIcgyqioPo"
    audio_file = download_youtube_audio(url)
    report = assess_clarity_final(audio_file)

    df_report = pd.DataFrame([report]).T
    df_report.columns = ["Value"]
    display(df_report)

except Exception as e:
    print(f"An error occurred: {e}")

System Running on: CPU


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Step 1: Transcribing audio to text...
Step 2: Analyzing Grammar & Structure...
Step 3: Evaluating Vocabulary Quality...


,Value
Candidate Speech (Sample),"Hi everyone, my name is singing glue and I'm a..."
Grammar Accuracy,99.1%
Grammar Issues Found,1
Common Errors,PUNCTUATION
Vocabulary Score,96.3/100
FINAL CLARITY INDEX,98.3/100
OVERALL ASSESSMENT,High Clarity (Professional & Structured)


In [14]:
import torch
import textstat
import yt_dlp
import pandas as pd
import language_tool_python
from transformers import pipeline

# ==============================
# 1. إعداد الجهاز وتحميل الموديلات
# ==============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"System Running on: {device.upper()}")

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=device,
    generate_kwargs={"task": "transcribe"}
)

tool = language_tool_python.LanguageTool('en-US')

# ==============================
# 2. دوال مساعدة (Helper Functions)
# ==============================
def download_youtube_audio(url):
    print(f"Downloading from: {url}")
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'interview_audio.%(ext)s',
        'quiet': True,
        'no_warnings': True,
        # السطرين الجايين بيساعدوا في تخطي بعض حمايات يوتيوب
        'extractor_args': {'youtube': {'player_client': ['android', 'web']}},
        'nocheckcertificate': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.prepare_filename(info)

def transcribe_audio(audio_path):
    """تحويل الصوت إلى نص"""
    print("Transcribing audio to text...")
    result = asr_pipeline(audio_path, return_timestamps=True)
    return result["text"].strip()

# ==============================
# 3. دالة الجرامر المستقلة (المعدلة)
# ==============================
def ai_function_grammar(text):
    """
    تأخذ النص وترجع قائمة بأخطاء الجرامر فقط مع تفاصيلها.
    """
    matches = tool.check(text)
    mistakes = []

    for match in matches:
        # الشرط ده هيتأكد إن الخطأ تصنيفه GRAMMAR فقط
        if match.category == 'GRAMMAR':
            mistakes.append({
                "category": match.category,
                "error_message": match.message,
                "incorrect_text": match.context,
                "suggestions": match.replacements[:3]  # أول 3 مقترحات للتعديل
            })

    return mistakes

def calculate_vocab_score(text):
    """حساب تقييم المفردات"""
    vocab_raw = textstat.dale_chall_readability_score(text)
    return min(vocab_raw * 10, 100)

# ==============================
# 4. الدالة المجمعة (The Orchestrator)
# ==============================
def process_interview_speech(audio_path):
    """
    هذه الدالة تجمع كل الخطوات السابقة في مكان واحد وترجع الناتج مهيكل
    """
    # 1. استخراج النص
    text = transcribe_audio(audio_path)
    if not text:
        return {"Status": "Error", "Message": "No clear speech detected."}

    # 2. تحليل الجرامر باستخدام الدالة الجديدة
    print("Analyzing Grammar & Structure...")
    grammar_mistakes = ai_function_grammar(text)

    # 3. حساب الدرجات
    print("Evaluating Vocabulary & Final Scores...")
    num_errors = len(grammar_mistakes)
    word_count = max(len(text.split()), 1)

    grammar_score = max(0, 100 - (num_errors / word_count * 100))
    vocab_score = calculate_vocab_score(text)
    final_score = (grammar_score * 0.7) + (vocab_score * 0.3)

    # 4. تحديد المستوى
    if final_score >= 85:
        level = "High Clarity (Professional & Structured)"
    elif final_score >= 70:
        level = "Moderate Clarity (Understandable but has minor issues)"
    else:
        level = "Low Clarity (Unstructured or many language errors)"

    # 5. شكل الناتج النهائي (Structured JSON-like format)
    return {
        "transcription_preview": text[:150] + "..." if len(text) > 150 else text,
        "word_count": word_count,
        "metrics": {
            "grammar_score": round(grammar_score, 1),
            "vocabulary_score": round(vocab_score, 1),
            "final_clarity_index": round(final_score, 1)
        },
        "assessment_level": level,
        "total_grammar_mistakes": num_errors,
        "grammar_mistakes_list": grammar_mistakes # القائمة المطلوبة
    }

# ==============================
# 5. تشغيل التجربة
# ==============================
if __name__ == "__main__":
    try:
        url = "https://www.youtube.com/watch?v=zaedfpFvn2w"
        audio_file = download_youtube_audio(url)

        # استدعاء الدالة المجمعة
        report = process_interview_speech(audio_file)

        # عرض النتائج بشكل منظم
        import json
        print("\n--- FINAL REPORT ---")
        print(json.dumps(report, indent=4, ensure_ascii=False))

    except Exception as e:
        print(f"An error occurred: {e}")

System Running on: CPU


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Transcribing audio to text...


Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Analyzing Grammar & Structure...
Evaluating Vocabulary & Final Scores...

--- FINAL REPORT ---
{
    "transcription_preview": "The court of profits are higher than productions. Well, I suppose that just means that people are doing the project. Gentlemen, Jean-François Charles ...",
    "word_count": 495,
    "metrics": {
        "grammar_score": 99.6,
        "vocabulary_score": 76.1,
        "final_clarity_index": 92.5
    },
    "assessment_level": "High Clarity (Professional & Structured)",
    "total_grammar_mistakes": 2,
    "grammar_mistakes_list": [
        {
            "category": "GRAMMAR",
            "error_message": "Possible agreement error.",
            "incorrect_text": "The court of profits are higher than productions. Well, I suppos...",
            "suggestions": [
                "is"
            ]
        },
        {
            "category": "GRAMMAR",
            "error_message": "After ‘It’, use the third-person verb form “makes”.",
            "incorrect_text":